In [19]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

In [20]:
df = pd.read_csv('../data/train.csv')
df_test = pd.read_csv('../data/test.csv')

In [21]:
X = df.drop(['Churn','CustomerID'], axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Calculate class imbalance from training data
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print("Scale pos weight:", scale_pos_weight)

Scale pos weight: 4.517866742113453


In [22]:
# Convert all your string/object columns first
categorical_cols = X_train.select_dtypes(include=['object']).columns
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')
    df_test[col] = df_test[col].astype('category')  

X_test.to_csv('X_test_xgb.csv', index=False)
y_test.to_csv('y_test_xgb.csv', index=False)
df_test.to_csv('test_xgb.csv', index=False)

C:\Users\kumar\AppData\Local\Temp\ipykernel_8400\4041122943.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=['object']).columns


In [23]:
xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    enable_categorical=True
)

xgb_model.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [24]:
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

y_pred_xgb = (y_prob_xgb >= 0.50).astype(int)

In [25]:
X_test

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled
121976,82,15.211698,1247.359265,Premium,Credit card,No,Both,No,Computer,21.365122,10.905371,16,Drama,1.564454,3,Female,5,Yes,No
232925,47,12.490732,587.064385,Premium,Mailed check,Yes,Movies,Yes,Tablet,3.241988,5.658260,40,Comedy,2.891674,5,Female,13,No,Yes
61236,113,13.057703,1475.520435,Standard,Mailed check,Yes,TV Shows,No,Mobile,23.038332,127.349847,7,Drama,1.788752,8,Male,22,No,Yes
222253,31,9.669522,299.755172,Premium,Bank transfer,No,Both,No,Computer,2.939787,15.027302,48,Comedy,3.665060,4,Male,10,No,No
176454,34,18.814475,639.692163,Standard,Electronic check,Yes,Movies,Yes,Tablet,1.998127,141.789040,28,Action,2.677158,6,Female,10,No,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110079,47,12.565654,590.585744,Standard,Credit card,No,Both,Yes,Mobile,16.417081,165.201421,31,Comedy,1.398329,9,Female,12,Yes,No
145223,21,14.579965,306.179262,Premium,Mailed check,No,Movies,No,Tablet,30.349524,33.598322,19,Comedy,4.290966,5,Female,0,Yes,Yes
159916,72,18.480613,1330.604144,Premium,Mailed check,Yes,Movies,Yes,Computer,34.445074,42.207345,42,Fantasy,4.369914,8,Male,17,Yes,No
229249,33,5.514994,181.994787,Premium,Mailed check,Yes,Movies,Yes,Computer,11.466072,157.819000,0,Sci-Fi,3.717602,0,Male,13,Yes,Yes


In [26]:
print(classification_report(y_test, y_pred_xgb))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))

print(confusion_matrix(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.91      0.68      0.78     39921
           1       0.32      0.68      0.44      8837

    accuracy                           0.68     48758
   macro avg       0.62      0.68      0.61     48758
weighted avg       0.80      0.68      0.72     48758

ROC-AUC: 0.7503863881873953
[[27305 12616]
 [ 2807  6030]]


XGBoost achieved a churn recall of 0.68 at the default threshold of 0.50, substantially outperforming the Random Forest baseline's churn recall of 0.30. However, churn precision was only 0.32, resulting in a relatively high number of false positives. The XGBoost model therefore shows stronger potential for identifying at-risk customers, but its precision-recall trade-off needs to be investigated through threshold tuning.

In [27]:
joblib.dump(xgb_model, "../models/xgboost.joblib")

['../models/xgboost.joblib']